In [1]:
# Production Process Analysis
# Step 1: Data Overview & Cleaning

In [10]:
import pandas as pd

df = pd.read_csv("../data/produktionsdaten_premium_5Jahre.csv")
df.head()
df.info()
df.describe()
df.columns
df.isnull().sum()
df.duplicated().sum()
df.dtypes
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8234 entries, 0 to 8233
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Datum                    8234 non-null   object 
 1   Unternehmen              8234 non-null   object 
 2   Produkt                  8234 non-null   object 
 3   Modifikation             8234 non-null   object 
 4   Produktionslinie         8234 non-null   object 
 5   Schicht                  8234 non-null   object 
 6   Stueckzahl               8234 non-null   int64  
 7   Ausschuss                8234 non-null   int64  
 8   Betriebsstunden          8234 non-null   float64
 9   Stillstandszeit_Min      8234 non-null   int64  
 10  MaxTemperatur            8234 non-null   float64
 11  Durchschnittstemperatur  8234 non-null   float64
 12  Softwareversion          8234 non-null   object 
 13  Firmwareversion          8234 non-null   object 
 14  EndOfLine_Test          

(8234, 21)

In [11]:

df.head().T
df. select_dtypes(include=["object"]).nunique()
list(df.columns)

['Datum',
 'Unternehmen',
 'Produkt',
 'Modifikation',
 'Produktionslinie',
 'Schicht',
 'Stueckzahl',
 'Ausschuss',
 'Betriebsstunden',
 'Stillstandszeit_Min',
 'MaxTemperatur',
 'Durchschnittstemperatur',
 'Softwareversion',
 'Firmwareversion',
 'EndOfLine_Test',
 'Materialkosten',
 'Energieverbrauch_kWh',
 'Auftragsnummer',
 'Status',
 'Fehlercode',
 'Mitarbeiter_Produktion']

In [3]:
# Datumsverarbeitung

df.filter(like="Datum").head()
df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce")
df["Datum"].dtype
df["Datum"].isnull().sum()

np.int64(0)

In [4]:
# Numerische Spalten prüfen

df.select_dtypes(include=["int64", "float64"]).head()
df.select_dtypes(include=["int64", "float64"]).describe()
df.select_dtypes(include=["int64", "float64"]).isnull().sum()
(df.select_dtypes(include=["int64", "float64"]) < 0).sum()

Stueckzahl                 0
Ausschuss                  0
Betriebsstunden            0
Stillstandszeit_Min        0
MaxTemperatur              0
Durchschnittstemperatur    0
Materialkosten             0
Energieverbrauch_kWh       0
Mitarbeiter_Produktion     0
dtype: int64

In [5]:
# Duplikate prüfen

df.duplicated().sum()
df.duplicated(subset=["Datum", "Produktionslinie", "Schicht"]).sum()
df[df.duplicated(subset=["Datum", "Produktionslinie", "Schicht"], keep=False)].head()

,Datum,Unternehmen,Produkt,Modifikation,Produktionslinie,Schicht,Stueckzahl,Ausschuss,Betriebsstunden,Stillstandszeit_Min,...,Durchschnittstemperatur,Softwareversion,Firmwareversion,EndOfLine_Test,Materialkosten,Energieverbrauch_kWh,Auftragsnummer,Status,Fehlercode,Mitarbeiter_Produktion
0,2019-01-01,Industrium GmbH,HMI-Terminal,HMI-10,Linie 3,Nacht,37,2,5.7,22,...,54.3,v4.9,f1.0,Bestanden,4947.15,7.09,A-24944,in Produktion,I/O_ERROR,90
4,2019-01-01,NextFactory Solutions,Edge Controller,EC-200,Linie 3,Nacht,285,4,9.4,57,...,43.6,v3.1,f2.3,Bestanden,9948.61,21.70,A-44437,in Produktion,POWER_FAIL,110
8,2019-01-03,Industrium GmbH,Monitoring Unit,MON-800,Linie 4,Nacht,120,1,10.0,9,...,41.2,v1.9,f1.2,Bestanden,4433.61,33.26,A-27115,pausiert,VIBRATION,90
9,2019-01-03,Industrium GmbH,Control Panel,CP-430,Linie 2,Spät,206,11,6.5,53,...,57.1,v4.1,f3.3,Bestanden,6256.42,21.71,A-36761,in Produktion,TEMP_HIGH,90
10,2019-01-03,TechSystems AG,Robust Panel PC,RPPC-1000,Linie 2,Spät,64,1,7.8,39,...,44.7,v2.7,f3.7,Bestanden,2100.19,12.05,A-12191,abgeschlossen,-,120


In [6]:
# Fehlende Werte analysieren

df.isnull().sum()
(df.isnull().mean() * 100).round(2)
df.isnull().sum()[df.isnull().sum() > 0]
df[df.isnull().any(axis=1)].head()

,Datum,Unternehmen,Produkt,Modifikation,Produktionslinie,Schicht,Stueckzahl,Ausschuss,Betriebsstunden,Stillstandszeit_Min,...,Durchschnittstemperatur,Softwareversion,Firmwareversion,EndOfLine_Test,Materialkosten,Energieverbrauch_kWh,Auftragsnummer,Status,Fehlercode,Mitarbeiter_Produktion


In [7]:
# Fehlende Werte fachlich behandeln

df["Ausschuss"] = df["Ausschuss"].fillna(0)
df["Stillstandszeit_Min"] = df["Stillstandszeit_Min"].fillna(0)
df["Energieverbrauch_kWh"] = df["Energieverbrauch_kWh"].fillna(df["Energieverbrauch_kWh"].median())
df.isnull().sum()

Datum                      0
Unternehmen                0
Produkt                    0
Modifikation               0
Produktionslinie           0
Schicht                    0
Stueckzahl                 0
Ausschuss                  0
Betriebsstunden            0
Stillstandszeit_Min        0
MaxTemperatur              0
Durchschnittstemperatur    0
Softwareversion            0
Firmwareversion            0
EndOfLine_Test             0
Materialkosten             0
Energieverbrauch_kWh       0
Auftragsnummer             0
Status                     0
Fehlercode                 0
Mitarbeiter_Produktion     0
dtype: int64

In [8]:
# Datentypen final prüfen

df.dtypes
df.memory_usage(deep=True)
df["Stueckzahl"] = df["Stueckzahl"].astype("int64")
df["Schicht"] = df["Schicht"].astype("category")
df.memory_usage(deep=True)

Index                         132
Datum                       65872
Unternehmen                534618
Produkt                    516933
Modifikation               458634
Produktionslinie           461104
Schicht                      8546
Stueckzahl                  65872
Ausschuss                   65872
Betriebsstunden             65872
Stillstandszeit_Min         65872
MaxTemperatur               65872
Durchschnittstemperatur     65872
Softwareversion            436402
Firmwareversion            436402
EndOfLine_Test             481154
Materialkosten              65872
Energieverbrauch_kWh        65872
Auftragsnummer             461104
Status                     485456
Fehlercode                 466433
Mitarbeiter_Produktion      65872
dtype: int64

In [9]:
# Abschlusskontrolle

df.isnull().sum()
df.dtypes
df.head()
df.shape

(8234, 21)